# Colab 01 - Build Embeddings / Qdrant Index

Run this notebook on Colab GPU to build the real Qdrant index with BioMedBERT large and BioCLIP.

Required Colab Secrets:
- `QDRANT_URL`
- `QDRANT_API_KEY`

Data must be uploaded to `MyDrive/KS_Project_2/data` before running the indexing cells.

## 1. Clone / update project

This cell always lands in `/content/project-ks2` and verifies the repository layout.

In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
from pathlib import Path

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git pull --ff-only || true

root = Path.cwd()
print("cwd:", root)
print("pyproject exists:", Path("pyproject.toml").exists())
print("medical_rag exists:", Path("src/medical_rag").exists())
assert Path("pyproject.toml").exists(), "Wrong folder: pyproject.toml not found"
assert Path("src/medical_rag").exists(), "Wrong folder: src/medical_rag not found"


## 2. Install dependencies

Editable install keeps local source changes active in Colab.

In [ ]:
!python -m pip install -U pip
!python -m pip install -e ".[gpu,qdrant,agent,eval]"
!python -m pip install -U requests accelerate bitsandbytes qwen-vl-utils huggingface_hub
!python -c "import medical_rag; print('medical_rag import OK')"


## 3. Configure secrets and model defaults

Secrets are read from Colab Secrets. BioMedBERT uses the correct public large checkpoint.

In [ ]:
import os

try:
    from google.colab import userdata
    for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY", "HF_TOKEN"]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("BIOMEDBERT_MODEL", "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract")
os.environ.setdefault("BIOMEDBERT_DIM", "1024")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("BIOMEDBERT_MODEL:", os.environ.get("BIOMEDBERT_MODEL"))
print("BIOMEDBERT_DIM:", os.environ.get("BIOMEDBERT_DIM"))

assert os.environ.get("QDRANT_URL"), "Missing QDRANT_URL in Colab Secrets"
assert os.environ.get("QDRANT_API_KEY"), "Missing QDRANT_API_KEY in Colab Secrets"


## 4. Compatibility patch for older commits

This is idempotent and prevents Colab from using the old invalid `large-uncased-abstract-fulltext` id.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("BIOMEDBERT_MODEL", "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract")
os.environ.setdefault("BIOMEDBERT_DIM", "1024")

p = Path("src/medical_rag/models/biomedbert.py")
text = p.read_text(encoding="utf-8")
text = text.replace(
    'import logging
from typing import Any',
    'import logging
import os
from typing import Any',
)
text = text.replace(
    'DEFAULT_MODEL_NAME = (
    "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract-fulltext"
)
EMBEDDING_DIM = 1024',
    'DEFAULT_MODEL_NAME = os.environ.get(
    "BIOMEDBERT_MODEL",
    "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract",
)
EMBEDDING_DIM = int(os.environ.get("BIOMEDBERT_DIM", "1024"))',
)
p.write_text(text, encoding="utf-8")

!python -m py_compile src/medical_rag/models/biomedbert.py
sys.path.insert(0, str(Path.cwd() / "src"))
from medical_rag.models.biomedbert import DEFAULT_MODEL_NAME, EMBEDDING_DIM
print("DEFAULT_MODEL_NAME =", DEFAULT_MODEL_NAME)
print("EMBEDDING_DIM =", EMBEDDING_DIM)
assert DEFAULT_MODEL_NAME == "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract"
assert EMBEDDING_DIM == 1024


## Data access from Google Drive

Upload local data to `/content/drive/MyDrive/KS_Project_2/data`.

This notebook uses a symlink instead of copying the whole dataset into `/content`.
That is safer for Colab Free because runtime resets do not delete Drive data.

In [ ]:
from pathlib import Path
import shutil

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/KS_Project_2/data")
LOCAL_DATA_DIR = Path("/content/project-ks2/data")

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped/unavailable:", exc)

print("Drive data dir:", DRIVE_DATA_DIR)
print("Drive data exists:", DRIVE_DATA_DIR.exists())
assert DRIVE_DATA_DIR.exists(), f"Missing {DRIVE_DATA_DIR}. Upload data to MyDrive/KS_Project_2/data first."

# Use symlink instead of copying huge data into ephemeral /content.
# If Colab runtime resets, rerun this cell after mounting Drive.
if LOCAL_DATA_DIR.exists() or LOCAL_DATA_DIR.is_symlink():
    if LOCAL_DATA_DIR.is_symlink():
        LOCAL_DATA_DIR.unlink()
    elif LOCAL_DATA_DIR.is_dir():
        shutil.rmtree(LOCAL_DATA_DIR)
    else:
        LOCAL_DATA_DIR.unlink()

LOCAL_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.symlink_to(DRIVE_DATA_DIR, target_is_directory=True)

print("Local data link:", LOCAL_DATA_DIR, "->", LOCAL_DATA_DIR.resolve())
files = [p for p in LOCAL_DATA_DIR.rglob("*") if p.is_file()]
print("data file count:", len(files))
for p in files[:50]:
    print(" -", p)
assert files, "Drive data folder is empty"


## Verify data availability

Qdrant indexing/evaluation needs `data/` to exist and contain dataset files.

In [ ]:
from pathlib import Path

data_dir = Path("data")
print("cwd:", Path.cwd())
print("data exists:", data_dir.exists())
assert data_dir.exists(), "Missing data/. Run the Drive data sync cell first."

files = [p for p in data_dir.rglob("*") if p.is_file()]
print("file count:", len(files))
for p in files[:80]:
    print(" -", p)
assert files, "data/ exists but contains no files"

patterns = ["*.json", "*.jsonl", "*.csv", "*.parquet", "*.jpg", "*.jpeg", "*.png", "*.pkl", "*.joblib"]
for pat in patterns:
    matches = list(data_dir.rglob(pat))
    print(f"data/**/*{pat[1:]}", len(matches))


## Verify GPU and HuggingFace model id

In [ ]:
import os
import torch
from huggingface_hub import model_info

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
model_id = os.environ["BIOMEDBERT_MODEL"]
info = model_info(model_id, token=os.environ.get("HF_TOKEN") or None)
print("BioMedBERT model OK:", info.modelId)


## Verify Qdrant Cloud auth before long indexing

In [ ]:
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


## Optional encoder smoke test

If this is too slow, skip it and go directly to indexing. The expected BioMedBERT shape for large is `[1024]`.

In [ ]:
!python -m medical_rag test-encoders --no-mock --include-bge


## Build real Qdrant index - small first

Use `--recreate` because BioMedBERT large requires text vector dimension 1024.

In [ ]:
!python -m medical_rag build-qdrant-index   --data-dir data   --qdrant-url "$QDRANT_URL"   --use-cloud-auth   --limit 100   --recreate   --no-use-mock-models


## Verify point counts after indexing

Expected after successful small run: `text_chunks.points > 0` and/or `image_patches.points > 0`.

In [ ]:
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


## Full indexing after small run succeeds

Only run after the small `--limit 100` run has non-zero Qdrant points. Remove `--limit` for full indexing.

In [ ]:
# !python -m medical_rag build-qdrant-index #   --data-dir data #   --qdrant-url "$QDRANT_URL" #   --use-cloud-auth #   --recreate #   --no-use-mock-models
